# NB10 — EVALUATION3 overlap audit

Notebook này chỉ phát hiện overlap giữa ảnh EVALUATION3 và các item từng được đưa vào scorer. Nó chưa chạy metric EVALUATION3 và không cần mapping `Cmt` 1/2/3.

## 1. Runtime portable

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    explicit = os.environ.get('FASHION_PROJECT_ROOT')
    candidates = [Path(explicit).expanduser()] if explicit else [start, *start.parents]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'src/evaluation/evaluation3_overlap.py').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy repo; hãy set FASHION_PROJECT_ROOT.')

REPO_ROOT = find_repo_root()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements-evaluation.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths
from src.evaluation.evaluation3_overlap import run_overlap_audit
RUNTIME_PATHS = load_runtime_paths(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)
print('ARTIFACT_ROOT:', RUNTIME_PATHS.artifact_root)

## 2. Chỉnh các đường dẫn EVALUATION3

`E3_ROOT` phải là thư mục chứa các folder outfit ID; mỗi folder có `U`, `B`, `S`, `G`. Không lọc `A-Test2000` ở đây vì workbook hiện tại chỉ có 30 dòng mang tag đó, không phải 2.000 dòng.

In [ ]:
E3_DIR = RUNTIME_PATHS.artifact_root / 'evaluation3'
E3_ROOT = E3_DIR / 'outfit'
CMT_FILE = E3_DIR / 'Cmt_ALL_20190325.xlsx'
ATTRIBUTE_FILE = E3_DIR / 'Attribute_ALL_UBSGsimple.xlsx'
SCORER_DIR = RUNTIME_PATHS.scorer_ready_dir
OUTPUT_DIR = RUNTIME_PATHS.artifact_root / 'evaluation3_overlap_audit'

required = [
    E3_ROOT, CMT_FILE, ATTRIBUTE_FILE,
    SCORER_DIR / 'scorer_ready_v2_train.jsonl',
    SCORER_DIR / 'scorer_ready_v2_valid.jsonl',
    SCORER_DIR / 'scorer_ready_v2_test.jsonl',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Thiếu input:\n- ' + '\n- '.join(missing))
print('Inputs OK')

## 3. Chạy audit

Mặc định lấy ảnh Polyvore trực tiếp từ `codewaly/polyvore1000`. Nếu đã có thư mục ảnh local theo `item_id`, đặt `POLYVORE_IMAGE_ROOT` thành đường dẫn đó để tránh tải lại.

In [ ]:
POLYVORE_IMAGE_ROOT = None
POLYVORE_HF_DATASET = None if POLYVORE_IMAGE_ROOT else 'codewaly/polyvore1000'

summary, output_paths = run_overlap_audit(
    evaluation3_root=E3_ROOT,
    development_split_paths={
        'train': SCORER_DIR / 'scorer_ready_v2_train.jsonl',
        'valid': SCORER_DIR / 'scorer_ready_v2_valid.jsonl',
        'test': SCORER_DIR / 'scorer_ready_v2_test.jsonl',
    },
    output_dir=OUTPUT_DIR,
    polyvore_image_root=POLYVORE_IMAGE_ROOT,
    polyvore_hf_dataset=POLYVORE_HF_DATASET,
    annotations_path=CMT_FILE,
    annotation_sheet='CMT',
    metadata_path=ATTRIBUTE_FILE,
    metadata_sheet='Num',
    model_development_splits={'train', 'valid'},
    near_hamming_threshold=4,
)

## 4. Đọc kết quả

Chỉ dùng manifest clean khi `status = PASS`. `strict_clean` loại overlap với cả Polyvore test; `model_clean` chỉ loại overlap với train/valid.

In [ ]:
import json
print(json.dumps(summary, ensure_ascii=False, indent=2))
for name, path in output_paths.items():
    print(f'{name}: {path}')